# Figure 5: Single-cell Perturbation and Mutation Prediction

Reads the single-cell run outputs from `Runs/SingleCell/output/` (`plot_inputs/` and `runs/`) and writes panels to `Figures/Figure5/output/`.

The plotting functions live in `figure5_helpers.py`; shared style constants live in `constants.py`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
FIGURES_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = FIGURES_DIR.parent
for path in (NOTEBOOK_DIR, FIGURES_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from figure5_helpers import (
    CMAP_BEIGE_PURPLE,
    Figure5Paths,
    setup_figure5_style,
    plot_auprc_boxplot,
    plot_adamson_10x005_pr_curves,
    plot_adamson_10x005_genotype_confusion,
    plot_zhao_pr_curves,
    plot_tian_top10_heatmap,
    plot_frangieh_scatter,
    plot_ccle_tmb_scatter,
    plot_metric_bars,
    plot_ccle_multitask_gene_bars,
    load_embedding_run,
    centroid_distance_summary,
    select_perturbations_from_clusters,
    build_fold_projection_tables,
    plot_centroid_dendrogram,
    plot_projection_per_fold,
    perturbation_coclustering_matrix,
    plot_coclustering_matrix,
    plot_selected_cluster_heatmap,
    summarize_cluster_purity_vs_auprc_one_fold,
    plot_cluster_purity_vs_auprc_one_fold,
    plot_leiden_composition_heatmap_for_fold,
    plot_selected_umap_perturbation_legend,
    head_weight_distance_summary,
    build_head_weight_projection_tables,
    plot_head_weight_projection_per_fold,
    head_weight_leiden_coclustering_matrix,
    plot_head_weight_leiden_coclustering_matrix,
)

setup_figure5_style()

paths = Figure5Paths(PROJECT_ROOT)
PLOT_INPUTS = paths.plot_input_root
RUNS_ROOT = paths.runs_root
FIGURE_ROOT = paths.figure_root
SUPPLEMENT_ROOT = FIGURE_ROOT / "Supplemental"
EMBEDDING_ROOT = paths.embedding_root

FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
SUPPLEMENT_ROOT.mkdir(parents=True, exist_ok=True)
EMBEDDING_ROOT.mkdir(parents=True, exist_ok=True)

PROJECTION_METHOD = "scanpy_umap"
MAX_POINTS_PER_FOLD = None

print(f"Project root : {PROJECT_ROOT}")
print(f"Plot inputs  : {PLOT_INPUTS}")
print(f"Run inputs   : {RUNS_ROOT}")
print(f"Figure output: {FIGURE_ROOT}")

## Perturb-seq AUPRC Boxplots

In [ ]:
plot_auprc_boxplot(
    PLOT_INPUTS / "Replogle_RPE1",
    "Replogle RPE1",
    FIGURE_ROOT,
    top_n=12,
    figsize=(2.9, 2.32),
)

plot_auprc_boxplot(
    PLOT_INPUTS / "Replogle_K562",
    "Replogle K562 essential",
    FIGURE_ROOT,
    top_n=12,
    figsize=(2.9, 2.3),
)

plot_auprc_boxplot(
    PLOT_INPUTS / "Adamson_10X010",
    "Adamson et al.",
    FIGURE_ROOT,
    top_n=12,
    figsize=(2.9, 2.5),
)

plot_auprc_boxplot(
    PLOT_INPUTS / "McFarland",
    "McFarland (cell-line CV)",
    FIGURE_ROOT,
    top_n=12,
    figsize=(2.9, 2.6),
)

## Zhao Drug Response PR Curves

In [ ]:
plot_zhao_pr_curves(
    PLOT_INPUTS / "Zhao" / "pr_curve_points_selected_targets.csv",
    FIGURE_ROOT,
    figsize=(3.0, 2.31),
)

## Tian CRISPRa Two-guide Heatmap

In [ ]:
plot_tian_top10_heatmap(
    PLOT_INPUTS / "Tian" / "directional_metrics_per_gene.csv",
    FIGURE_ROOT,
    top_n=10,
    figsize=(2.6, 2.72)
)

## Frangieh Transfer and Per-gene Panels

In [ ]:
plot_frangieh_scatter(
    PLOT_INPUTS / "Frangieh" / "metric_summary_per_gene_folds.csv",
    FIGURE_ROOT,
    figsize=(2.9, 2.6),
)

frangieh_baseline = PLOT_INPUTS / "Frangieh_per_gene"

plot_metric_bars(
    frangieh_baseline / "metric_summary_per_gene.csv",
    frangieh_baseline / "metric_summary_per_gene_folds.csv",
    FIGURE_ROOT,
    output_prefix="frangieh_per_gene",
    metric_col="oof_roc_auc",
    fold_metric_col="roc_auc",
    metric_label="ROC AUC",
    rank_mode="metric",
    figsize=(2.8, 2.7)
)

plot_metric_bars(
    frangieh_baseline / "metric_summary_per_gene.csv",
    frangieh_baseline / "metric_summary_per_gene_folds.csv",
    FIGURE_ROOT,
    output_prefix="frangieh_per_gene",
    metric_col="oof_average_precision",
    fold_metric_col="average_precision",
    metric_label="AUPRC",
    rank_mode="metric",
    figsize=(2.8, 2.7)
)

plot_metric_bars(
    frangieh_baseline / "metric_summary_per_gene.csv",
    frangieh_baseline / "metric_summary_per_gene_folds.csv",
    FIGURE_ROOT,
    output_prefix="frangieh_per_gene",
    metric_col="oof_average_precision",
    fold_metric_col="average_precision",
    metric_label="AUPRC",
    rank_mode="fold_increase",
    figsize=(2.8, 2.7)
)

## CCLE TMB Prediction

In [ ]:
ccle_dir = PLOT_INPUTS / "CCLE"

plot_ccle_tmb_scatter(ccle_dir, FIGURE_ROOT, level="cell-line", booster="xgboost")
plot_ccle_tmb_scatter(ccle_dir, FIGURE_ROOT, level="cell", booster="xgboost")

## CCLE Multitask Mutation Prediction

In [ ]:
ccle_multitask = ccle_dir / "multitask_fold_metrics_150.csv"

plot_ccle_multitask_gene_bars(
    ccle_multitask,
    FIGURE_ROOT,
    metric_col="auprc",
    metric_label="AUPRC",
    top_n=20,
    rank_mode="metric",
    figsize=(5, 2.8)
)

plot_ccle_multitask_gene_bars(
    ccle_multitask,
    FIGURE_ROOT,
    metric_col="roc_auc",
    metric_label="ROC AUC",
    top_n=20,
    rank_mode="metric",
    figsize=(5, 2.8)
)

plot_ccle_multitask_gene_bars(
    ccle_multitask,
    FIGURE_ROOT,
    metric_col="auprc",
    metric_label="AUPRC",
    top_n=20,
    rank_mode="fold_increase",
    figsize=(5, 2.8)
)

## K562 Embedding Dendrogram and UMAP Panels

In [ ]:
# Leiden clusters were chosen according to the local run. Different runs can renumber clusters, so adjust these IDs if needed.
K562_SELECTED_LEIDEN_CLUSTERS = [3, 5, 6, 7, 8, 9, 10,12,13, 14, 15, 16, 17, 18]

k562_mapping, k562_embeddings, k562_metrics, k562_fold_metrics = load_embedding_run(
    RUNS_ROOT / "ReplogleWeissman2022_K562_essential_mc300_simple_wide_run"
)
k562_summary = centroid_distance_summary(k562_mapping, k562_embeddings, n_clusters=15)
k562_selected = select_perturbations_from_clusters(k562_summary, excluded_clusters=(12,))

plot_centroid_dendrogram(
    k562_summary,
    EMBEDDING_ROOT / "K562_essential_centroid_euclidean_distance_dendrogram_colorbars.pdf",
)

k562_projection_tables, k562_projection_method = build_fold_projection_tables(
    k562_mapping,
    k562_embeddings,
    k562_fold_metrics,
    method=PROJECTION_METHOD,
    max_points_per_fold=MAX_POINTS_PER_FOLD,
)

plot_projection_per_fold(
    k562_projection_tables,
    EMBEDDING_ROOT / f"K562_essential_{k562_projection_method}_perturbation_per_fold.pdf",
    method_used=k562_projection_method,
    color_mode="perturbation",
    selected_perturbations=k562_selected,
)
plot_selected_umap_perturbation_legend(
    k562_projection_tables,
    EMBEDDING_ROOT / "K562_essential_umap_selected_perturbations_legend.pdf",
    selected_perturbations=k562_selected,
    ncol=16,
)
plot_projection_per_fold(
    k562_projection_tables,
    EMBEDDING_ROOT / f"K562_essential_{k562_projection_method}_leiden_per_fold.pdf",
    method_used=k562_projection_method,
    color_mode="leiden",
)
plot_projection_per_fold(
    k562_projection_tables,
    EMBEDDING_ROOT / f"K562_essential_{k562_projection_method}_auprc_per_fold.pdf",
    method_used=k562_projection_method,
    color_mode="auprc",
)

k562_coclustering = perturbation_coclustering_matrix(k562_projection_tables)
plot_coclustering_matrix(
    k562_coclustering,
    k562_summary,
    EMBEDDING_ROOT / "K562_essential_coclustering_matrix_with_dendrogram_colorbars.pdf",
    selected_perturbations=k562_selected,
)
plot_leiden_composition_heatmap_for_fold(
    k562_projection_tables,
    k562_summary,
    k562_selected,
    EMBEDDING_ROOT / f"K562_essential_fold_1_leiden_composition_with_dendrogram_colorbars.pdf",
    fold=1,
    )

plot_selected_cluster_heatmap(
    k562_projection_tables,
    k562_selected,
    k562_summary,
    EMBEDDING_ROOT / "K562_essential_fold_1_selected_perturbations_selected_leiden_heatmap.pdf",
    fold=1,
    selected_leiden_clusters=K562_SELECTED_LEIDEN_CLUSTERS,
)

## RPE1 Embedding Dendrogram and UMAP Panels

In [ ]:
# Leiden clusters were chosen according to the local run. Different runs can renumber clusters, so adjust these IDs if needed.
RPE1_SELECTED_LEIDEN_CLUSTERS = [13, 2, 10, 11, 14, 4, 5, 12, 15, 16]
rpe1_mapping, rpe1_embeddings, rpe1_metrics, rpe1_fold_metrics = load_embedding_run(
    RUNS_ROOT / "ReplogleWeissman2022_rpe1_mc300_simple_wide_run"
)
rpe1_summary = centroid_distance_summary(rpe1_mapping, rpe1_embeddings, n_clusters=15)
rpe1_selected = select_perturbations_from_clusters(rpe1_summary, excluded_clusters=(9,))

plot_centroid_dendrogram(
    rpe1_summary,
    EMBEDDING_ROOT / "rpe1_centroid_euclidean_distance_dendrogram_colorbars.pdf",
)

rpe1_projection_tables, rpe1_projection_method = build_fold_projection_tables(
    rpe1_mapping,
    rpe1_embeddings,
    rpe1_fold_metrics,
    method=PROJECTION_METHOD,
    max_points_per_fold=MAX_POINTS_PER_FOLD,
)

plot_projection_per_fold(
    rpe1_projection_tables,
    EMBEDDING_ROOT / f"rpe1_{rpe1_projection_method}_perturbation_per_fold.pdf",
    method_used=rpe1_projection_method,
    color_mode="perturbation",
    selected_perturbations=rpe1_selected,
)
plot_selected_umap_perturbation_legend(
    rpe1_projection_tables,
    EMBEDDING_ROOT / "rpe1_umap_selected_perturbations_legend.pdf",
    selected_perturbations=rpe1_selected,
    ncol=16,
)
plot_projection_per_fold(
    rpe1_projection_tables,
    EMBEDDING_ROOT / f"rpe1_{rpe1_projection_method}_leiden_per_fold.pdf",
    method_used=rpe1_projection_method,
    color_mode="leiden",
)
plot_projection_per_fold(
    rpe1_projection_tables,
    EMBEDDING_ROOT / f"rpe1_{rpe1_projection_method}_auprc_per_fold.pdf",
    method_used=rpe1_projection_method,
    color_mode="auprc",
)
rpe1_coclustering = perturbation_coclustering_matrix(rpe1_projection_tables)
plot_coclustering_matrix(
    rpe1_coclustering,
    rpe1_summary,
    EMBEDDING_ROOT / "rpe1_coclustering_matrix_with_dendrogram_colorbars.pdf",
    selected_perturbations=rpe1_selected,
)
plot_leiden_composition_heatmap_for_fold(
    rpe1_projection_tables,
    rpe1_summary,
    rpe1_selected,
    EMBEDDING_ROOT / f"rpe1_fold_1_leiden_composition_with_dendrogram_colorbars.pdf",
    fold=1,
)
plot_selected_cluster_heatmap(
    rpe1_projection_tables,
    rpe1_selected,
    rpe1_summary,
    EMBEDDING_ROOT / "rpe1_fold_1_selected_perturbations_selected_leiden_heatmap.pdf",
    fold=1,
    selected_leiden_clusters=RPE1_SELECTED_LEIDEN_CLUSTERS,
)

## Supplemental RPE1 Cluster Purity vs AUPRC


In [ ]:
rpe1_purity_long, rpe1_purity_summary, rpe1_purity_stats = summarize_cluster_purity_vs_auprc_one_fold(
    rpe1_projection_tables,
    fold=1,
    selected_leiden_clusters=RPE1_SELECTED_LEIDEN_CLUSTERS,
)

rpe1_purity_summary.to_csv(
    SUPPLEMENT_ROOT / "rpe1_fold1_cluster_purity_vs_auprc_summary.csv",
    index=False,
)
pd.DataFrame([rpe1_purity_stats]).to_csv(
    SUPPLEMENT_ROOT / "rpe1_fold1_cluster_purity_vs_auprc_stats.csv",
    index=False,
)

plot_cluster_purity_vs_auprc_one_fold(
    rpe1_purity_summary,
    rpe1_purity_stats,
    SUPPLEMENT_ROOT / "rpe1_fold1_cluster_purity_vs_auprc.pdf",
    title="RPE1 fold 1 cluster purity vs predictivity",
)

display(pd.DataFrame([rpe1_purity_stats]))
display(rpe1_purity_summary.sort_values("auprc", ascending=False).head(20))


## Adamson/Weissman Panels

In [ ]:
ADAMSON_WEISSMAN_RUNS = {
    "10X001": PLOT_INPUTS / "Adamson_10X001",
    "10X005": PLOT_INPUTS / "Adamson_10X005",
    "10X010": PLOT_INPUTS / "Adamson_10X010",
}

plot_auprc_boxplot(
    ADAMSON_WEISSMAN_RUNS["10X001"],
    "Adamson/Weissman 10X001",
    FIGURE_ROOT,
    top_n=7,
    figsize=(2.9, 2.35),
)

plot_auprc_boxplot(
    ADAMSON_WEISSMAN_RUNS["10X005"],
    "Adamson/Weissman 10X005",
    FIGURE_ROOT,
    top_n=3,
    figsize=(2.4, 2.35),
)

plot_adamson_10x005_pr_curves(
  ADAMSON_WEISSMAN_RUNS["10X005"],
  title="Adamson",
  figsize=(2.9, 2.18),
  save_path=FIGURE_ROOT / "adamson_10x005_pr_curves.pdf",
)
plot_adamson_10x005_genotype_confusion(
  ADAMSON_WEISSMAN_RUNS["10X005"],
  title="Adamson 10X005",
  threshold=0.5,
  figsize=(3.8, 3.3),
  save_path=FIGURE_ROOT / "adamson_10x005_genotype_confusion.pdf",
)

## Target Head-weight Structure

In [ ]:
k562_mapping, k562_embeddings, k562_metrics, k562_fold_metrics = load_embedding_run(
    RUNS_ROOT / "ReplogleWeissman2022_K562_essential_mc300_simple_wide_run"
)
rpe1_mapping, rpe1_embeddings, rpe1_metrics, rpe1_fold_metrics = load_embedding_run(
    RUNS_ROOT / "ReplogleWeissman2022_rpe1_mc300_simple_wide_run"
)

In [ ]:
HEAD_WEIGHT_ROOT = EMBEDDING_ROOT / "TargetHeadWeights"
HEAD_WEIGHT_ROOT.mkdir(parents=True, exist_ok=True)

HEAD_WEIGHT_DEFAULT_METRIC = "euclidean"
HEAD_WEIGHT_DEFAULT_N_NEIGHBORS = 2
HEAD_WEIGHT_DEFAULT_LEIDEN_RESOLUTION = 1.0

HEAD_WEIGHT_DATASETS = {
    "rpe1": {
        "run": RUNS_ROOT / "ReplogleWeissman2022_rpe1_mc300_simple_wide_run",
        "fold_metrics": rpe1_fold_metrics,
        "title": "RPE1",
        "metric": HEAD_WEIGHT_DEFAULT_METRIC,
        "n_neighbors": HEAD_WEIGHT_DEFAULT_N_NEIGHBORS,
        "leiden_resolution": HEAD_WEIGHT_DEFAULT_LEIDEN_RESOLUTION,
        "coclustering_figsize": (8, 8),
    },
    "K562_essential": {
        "run": RUNS_ROOT / "ReplogleWeissman2022_K562_essential_mc300_simple_wide_run",
        "fold_metrics": k562_fold_metrics,
        "title": "K562 essential",
        "metric": HEAD_WEIGHT_DEFAULT_METRIC,
        "n_neighbors": HEAD_WEIGHT_DEFAULT_N_NEIGHBORS,
        "leiden_resolution": HEAD_WEIGHT_DEFAULT_LEIDEN_RESOLUTION,
        "coclustering_figsize": (11, 11),
    },
}

for dataset_prefix, cfg in HEAD_WEIGHT_DATASETS.items():
    summary = head_weight_distance_summary(
        cfg["run"],
        metric=cfg.get("metric", HEAD_WEIGHT_DEFAULT_METRIC),
    )
    projections, projection_method = build_head_weight_projection_tables(
        cfg["run"],
        cfg["fold_metrics"],
        summary,
        metric=cfg.get("metric", HEAD_WEIGHT_DEFAULT_METRIC),
        n_neighbors=cfg.get("n_neighbors", HEAD_WEIGHT_DEFAULT_N_NEIGHBORS),
        leiden_resolution=cfg.get("leiden_resolution", HEAD_WEIGHT_DEFAULT_LEIDEN_RESOLUTION),
        min_dist=0.3,
    )

    pd.concat(projections.values(), ignore_index=True).to_csv(
        HEAD_WEIGHT_ROOT / f"{dataset_prefix}_head_weight_projection_per_fold.csv",
        index=False,
    )

    plot_head_weight_projection_per_fold(
        projections,
        HEAD_WEIGHT_ROOT / f"{dataset_prefix}_head_weight_umap_auprc_per_fold.pdf",
        method_used=projection_method,
        color_mode="auprc",
        label_top_n=None,
    )
    plot_head_weight_projection_per_fold(
        projections,
        HEAD_WEIGHT_ROOT / f"{dataset_prefix}_head_weight_umap_leiden_per_fold.pdf",
        method_used=projection_method,
        color_mode="leiden",
        label_top_n=0,
    )
    leiden_coclustering = head_weight_leiden_coclustering_matrix(projections)
    leiden_coclustering.to_csv(
        HEAD_WEIGHT_ROOT / f"{dataset_prefix}_head_weight_leiden_coclustering.csv"
    )
    plot_head_weight_leiden_coclustering_matrix(
        leiden_coclustering,
        HEAD_WEIGHT_ROOT / f"{dataset_prefix}_head_weight_leiden_coclustering_clustermap.pdf",
        title=f"{cfg['title']} target head-weight Leiden co-clustering",
        figsize=cfg.get("coclustering_figsize", (8, 8)),
    )